In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/equal-neutral-submit-version/train_equal.csv
/kaggle/input/equal-neutral-submit-version/test_equal.csv


In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
import matplotlib.pyplot as plt
import re

torch.manual_seed(42)
np.random.seed(42)

2024-08-26 02:15:26.868198: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-26 02:15:26.868314: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-26 02:15:26.996244: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
def preprocess_text(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE).lower()

def prepare_balanced_data(df, num_samples):
    min_class_count = min(num_samples, df['sentiment_equal'].value_counts().min())
    return pd.concat([df[df['sentiment_equal'] == sentiment].sample(min_class_count, random_state=42) 
                      for sentiment in df['sentiment_equal'].unique()], ignore_index=True)

In [4]:
def tokenize_function(examples, tokenizer, max_length):
    return tokenizer(examples['comment'], padding='max_length', truncation=True, max_length=max_length)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}
def print_distribution(df, name):
    print(f"\nDistribution for {name}:")
    print(df['sentiment_equal'].value_counts())
    print(df['sentiment_equal'].value_counts(normalize=True))

In [5]:
train_file = '/kaggle/input/equal-neutral-submit-version/train_equal.csv'
test_file = '/kaggle/input/equal-neutral-submit-version/test_equal.csv'
sample_sizes = [10000, 15000, 20000, 40000, 80000]

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

for df in [train_df, test_df]:
    df['comment'] = df['comment'].apply(preprocess_text)

sentiment_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
train_df['label'] = train_df['sentiment_equal'].map(sentiment_mapping)
test_df['label'] = test_df['sentiment_equal'].map(sentiment_mapping)

print_distribution(train_df, "Original Train Set")
print_distribution(test_df, "Original Test Set")



Distribution for Original Train Set:
sentiment_equal
Positive    1089167
Neutral      156360
Negative      86702
Name: count, dtype: int64
sentiment_equal
Positive    0.817552
Neutral     0.117367
Negative    0.065080
Name: proportion, dtype: float64

Distribution for Original Test Set:
sentiment_equal
Positive    272565
Neutral      39115
Negative     21378
Name: count, dtype: int64
sentiment_equal
Positive    0.818371
Neutral     0.117442
Negative    0.064187
Name: proportion, dtype: float64


In [6]:
tokenizer = AutoTokenizer.from_pretrained('distilroberta-base')
results = []

for size in sample_sizes:
    print(f"\nExperimenting with up to {size} samples per class")
    
    balanced_train_df = prepare_balanced_data(train_df, size)
    balanced_test_df = prepare_balanced_data(test_df, test_df['sentiment_equal'].value_counts().min())
    
    print_distribution(balanced_train_df, f"Balanced Train Set ({size} samples per class)")
    print_distribution(balanced_test_df, "Balanced Test Set")
    
    train_dataset = Dataset.from_pandas(balanced_train_df).map(
        lambda x: tokenize_function(x, tokenizer, max_length=128), batched=True)
    test_dataset = Dataset.from_pandas(balanced_test_df).map(
        lambda x: tokenize_function(x, tokenizer, max_length=128), batched=True)
    
    model = AutoModelForSequenceClassification.from_pretrained('distilroberta-base', num_labels=3)
    
    training_args = TrainingArguments(
        output_dir=f'./distilroberta_results_{size}_samples',
        num_train_epochs=4,
        per_device_train_batch_size=64,
        per_device_eval_batch_size=64,
        warmup_steps=500,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        metric_for_best_model="accuracy",
        report_to="none",
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    eval_results = trainer.evaluate()
    
    results.append(('distilroberta-base', size, eval_results['eval_accuracy'], eval_results['eval_f1']))
    
    print(f"Results for {size} samples per class:")
    for metric, value in eval_results.items():
        print(f"{metric}: {value:.4f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


Experimenting with up to 10000 samples per class

Distribution for Balanced Train Set (10000 samples per class):
sentiment_equal
Positive    10000
Negative    10000
Neutral     10000
Name: count, dtype: int64
sentiment_equal
Positive    0.333333
Negative    0.333333
Neutral     0.333333
Name: proportion, dtype: float64

Distribution for Balanced Test Set:
sentiment_equal
Positive    21378
Neutral     21378
Negative    21378
Name: count, dtype: int64
sentiment_equal
Positive    0.333333
Neutral     0.333333
Negative    0.333333
Name: proportion, dtype: float64


... (27 outputs left out)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.544900,0.531407,0.769810,0.768210,0.767687,0.769810
2,0.500600,0.526488,0.774472,0.769260,0.769842,0.774472
3,0.458200,0.538287,0.775439,0.774594,0.774135,0.775439
4,0.422400,0.545584,0.774363,0.772096,0.770946,0.774363


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead u

Results for 80000 samples per class:
eval_loss: 0.5456
eval_accuracy: 0.7744
eval_f1: 0.7721
eval_precision: 0.7709
eval_recall: 0.7744
eval_runtime: 127.5806
eval_samples_per_second: 502.6940
eval_steps_per_second: 3.9350
epoch: 4.0000


In [7]:
sizes = [result[1] for result in results]
accuracies = [result[2] for result in results]
f1_scores = [result[3] for result in results]

for model_name, size, accuracy, f1 in results:
    print(f"Model: {model_name}, Max samples per class: {size}, Accuracy: {accuracy:.4f}, F1 Score: {f1:.4f}")

Model: distilroberta-base, Max samples per class: 10000, Accuracy: 0.7590, F1 Score: 0.7570
Model: distilroberta-base, Max samples per class: 15000, Accuracy: 0.7610, F1 Score: 0.7593
Model: distilroberta-base, Max samples per class: 20000, Accuracy: 0.7640, F1 Score: 0.7620
Model: distilroberta-base, Max samples per class: 40000, Accuracy: 0.7693, F1 Score: 0.7681
Model: distilroberta-base, Max samples per class: 80000, Accuracy: 0.7744, F1 Score: 0.7721
